In [ ]:
# 03 Data Quality

In diesem Notebook werden die vorbereiteten Yellow-Taxi-Daten bereinigt.

Ziel:
- ungültige Zeitstempel entfernen
- Fahrten mit unplausibler Dauer entfernen
- Fahrten mit unplausibler Distanz entfernen
- negative oder unrealistische Beträge entfernen
- ungültige Passenger Counts entfernen
- bereinigten Datensatz als `/taxi/clean` speichern

Die Rohdaten bleiben unverändert. Die bereinigten Daten dienen später als Grundlage für die Analyse.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import subprocess

spark = SparkSession.builder \
    .appName("Taxi Data Quality") \
    .getOrCreate()

spark

26/05/19 14:57:37 WARN Utils: Your hostname, bdlc-021 resolves to a loopback address: 127.0.1.1; using 10.176.129.84 instead (on interface ens3)
26/05/19 14:57:37 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/19 14:57:38 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
PREPARED_PATH = "/taxi/prepared"
CLEAN_PATH = "/taxi/clean"

print("Prepared path:", PREPARED_PATH)
print("Clean path:", CLEAN_PATH)

Prepared path: /taxi/prepared
Clean path: /taxi/clean


26/05/19 14:57:53 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [ ]:
Prepared Daten laden

In [3]:
df = spark.read.parquet(PREPARED_PATH)

df.printSchema()

root
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance_miles: double (nullable = true)
 |-- pickup_location_id: long (nullable = true)
 |-- dropoff_location_id: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- pickup_year: integer (nullable = true)
 |-- pickup_month: integer (nullable = true)
 |-- pickup_hour: integer (nullable = true)
 |-- pickup_weekday: integer (nullable = true)
 |-- trip_duration_minutes: double (nullable = true)
 |-- trip_distance_km: double (nullable = true)
 |-- avg_speed_kmh: double (nullable = true)
 |-- tip_percentage: double (nullable = true)
 |-- file_year: integer (nullable = true)
 |-- file_month: integer (nullable = true)



In [4]:
#Records vor bereinigung
records_before = df.count()

print(f"Records before cleaning: {records_before}")

[Stage 1:======================================================>(159 + 1) / 160]

Records before cleaning: 633694594


In [ ]:
Übersicht der fehlerhaften werte

In [5]:
df.select(
    F.min("pickup_datetime").alias("min_pickup_datetime"),
    F.max("pickup_datetime").alias("max_pickup_datetime"),
    F.min("dropoff_datetime").alias("min_dropoff_datetime"),
    F.max("dropoff_datetime").alias("max_dropoff_datetime"),
    F.min("trip_duration_minutes").alias("min_duration"),
    F.max("trip_duration_minutes").alias("max_duration"),
    F.min("trip_distance_km").alias("min_distance_km"),
    F.max("trip_distance_km").alias("max_distance_km"),
    F.min("avg_speed_kmh").alias("min_speed_kmh"),
    F.max("avg_speed_kmh").alias("max_speed_kmh"),
    F.min("total_amount").alias("min_total_amount"),
    F.max("total_amount").alias("max_total_amount")
).show(truncate=False)

+-------------------+-------------------+--------------------+--------------------+--------------------+--------------------+------------------+-----------------+---------------------+-------------------+----------------+----------------+
|min_pickup_datetime|max_pickup_datetime|min_dropoff_datetime|max_dropoff_datetime|min_duration        |max_duration        |min_distance_km   |max_distance_km  |min_speed_kmh        |max_speed_kmh      |min_total_amount|max_total_amount|
+-------------------+-------------------+--------------------+--------------------+--------------------+--------------------+------------------+-----------------+---------------------+-------------------+----------------+----------------+
|2001-01-01 00:01:48|2098-09-11 02:23:31|1900-01-01 00:00:00 |2253-08-23 07:56:38 |-6.100172596666667E7|1.2537316083333333E8|-6.5725645801896E7|9.4977790010862E7|-7.887077496227519E10|4.955362957088452E9|-1871.8         |3950611.6       |
+-------------------+-------------------+---

In [ ]:
Fehlerhafte werte Zählen

In [6]:
quality_checks_before = df.agg(
    F.count("*").alias("total_records"),

    F.sum(F.when(F.col("pickup_year") != F.col("file_year"), 1).otherwise(0)).alias("invalid_pickup_year"),
    F.sum(F.when(F.col("pickup_month") != F.col("file_month"), 1).otherwise(0)).alias("invalid_pickup_month"),
    F.sum(F.when(F.col("dropoff_datetime") <= F.col("pickup_datetime"), 1).otherwise(0)).alias("dropoff_before_pickup"),

    F.sum(F.when(F.col("trip_duration_minutes") <= 0, 1).otherwise(0)).alias("duration_leq_0"),
    F.sum(F.when(F.col("trip_duration_minutes") > 180, 1).otherwise(0)).alias("duration_gt_180"),

    F.sum(F.when(F.col("trip_distance_km") <= 0, 1).otherwise(0)).alias("distance_leq_0"),
    F.sum(F.when(F.col("trip_distance_km") > 160, 1).otherwise(0)).alias("distance_gt_160"),

    F.sum(F.when(F.col("avg_speed_kmh") <= 0, 1).otherwise(0)).alias("speed_leq_0"),
    F.sum(F.when(F.col("avg_speed_kmh") > 150, 1).otherwise(0)).alias("speed_gt_150"),

    F.sum(F.when(F.col("passenger_count") <= 0, 1).otherwise(0)).alias("passenger_count_leq_0"),
    F.sum(F.when(F.col("passenger_count") > 6, 1).otherwise(0)).alias("passenger_count_gt_6"),

    F.sum(F.when(F.col("fare_amount") < 0, 1).otherwise(0)).alias("fare_amount_negative"),
    F.sum(F.when(F.col("tip_amount") < 0, 1).otherwise(0)).alias("tip_amount_negative"),
    F.sum(F.when(F.col("total_amount") <= 0, 1).otherwise(0)).alias("total_amount_leq_0"),
    F.sum(F.when(F.col("total_amount") > 1000, 1).otherwise(0)).alias("total_amount_gt_1000")
)

quality_checks_before.show(truncate=False)

[Stage 7:=====================================================> (157 + 3) / 160]

+-------------+-------------------+--------------------+---------------------+--------------+---------------+--------------+---------------+-----------+------------+---------------------+--------------------+--------------------+-------------------+------------------+--------------------+
|total_records|invalid_pickup_year|invalid_pickup_month|dropoff_before_pickup|duration_leq_0|duration_gt_180|distance_leq_0|distance_gt_160|speed_leq_0|speed_gt_150|passenger_count_leq_0|passenger_count_gt_6|fare_amount_negative|tip_amount_negative|total_amount_leq_0|total_amount_gt_1000|
+-------------+-------------------+--------------------+---------------------+--------------+---------------+--------------+---------------+-----------+------------+---------------------+--------------------+--------------------+-------------------+------------------+--------------------+
|633694594    |3546               |16297               |706257               |706257        |1133241        |4590920       |7501  

In [ ]:
In Prozenten

In [7]:
quality_percentages_before = quality_checks_before.select(
    F.col("total_records"),

    F.round(F.col("invalid_pickup_year") / F.col("total_records") * 100, 4).alias("invalid_pickup_year_pct"),
    F.round(F.col("invalid_pickup_month") / F.col("total_records") * 100, 4).alias("invalid_pickup_month_pct"),
    F.round(F.col("dropoff_before_pickup") / F.col("total_records") * 100, 4).alias("dropoff_before_pickup_pct"),

    F.round(F.col("duration_leq_0") / F.col("total_records") * 100, 4).alias("duration_leq_0_pct"),
    F.round(F.col("duration_gt_180") / F.col("total_records") * 100, 4).alias("duration_gt_180_pct"),

    F.round(F.col("distance_leq_0") / F.col("total_records") * 100, 4).alias("distance_leq_0_pct"),
    F.round(F.col("distance_gt_160") / F.col("total_records") * 100, 4).alias("distance_gt_160_pct"),

    F.round(F.col("speed_leq_0") / F.col("total_records") * 100, 4).alias("speed_leq_0_pct"),
    F.round(F.col("speed_gt_150") / F.col("total_records") * 100, 4).alias("speed_gt_150_pct"),

    F.round(F.col("passenger_count_leq_0") / F.col("total_records") * 100, 4).alias("passenger_count_leq_0_pct"),
    F.round(F.col("passenger_count_gt_6") / F.col("total_records") * 100, 4).alias("passenger_count_gt_6_pct"),

    F.round(F.col("fare_amount_negative") / F.col("total_records") * 100, 4).alias("fare_amount_negative_pct"),
    F.round(F.col("tip_amount_negative") / F.col("total_records") * 100, 4).alias("tip_amount_negative_pct"),
    F.round(F.col("total_amount_leq_0") / F.col("total_records") * 100, 4).alias("total_amount_leq_0_pct"),
    F.round(F.col("total_amount_gt_1000") / F.col("total_records") * 100, 4).alias("total_amount_gt_1000_pct")
)

quality_percentages_before.show(truncate=False)

[Stage 10:====================================================> (155 + 5) / 160]

+-------------+-----------------------+------------------------+-------------------------+------------------+-------------------+------------------+-------------------+---------------+----------------+-------------------------+------------------------+------------------------+-----------------------+----------------------+------------------------+
|total_records|invalid_pickup_year_pct|invalid_pickup_month_pct|dropoff_before_pickup_pct|duration_leq_0_pct|duration_gt_180_pct|distance_leq_0_pct|distance_gt_160_pct|speed_leq_0_pct|speed_gt_150_pct|passenger_count_leq_0_pct|passenger_count_gt_6_pct|fare_amount_negative_pct|tip_amount_negative_pct|total_amount_leq_0_pct|total_amount_gt_1000_pct|
+-------------+-----------------------+------------------------+-------------------------+------------------+-------------------+------------------+-------------------+---------------+----------------+-------------------------+------------------------+------------------------+-----------------------

In [ ]:
## Filterregeln

Die Spalte `trip_duration_minutes` wurde im Preprocessing aus der Differenz von Dropoff- und Pickup-Zeitpunkt berechnet. 
Da `unix_timestamp` Sekunden liefert und anschliessend durch 60 geteilt wurde, ist die Einheit Minuten.

Für die Analyse werden nur plausible Fahrten verwendet.

Verwendete Regeln:

- Wichtige Spalten dürfen nicht `NULL` sein
- Numerische Spalten dürfen nicht `NaN` sein
- Pickup-Jahr muss dem Dateijahr entsprechen
- Pickup-Monat muss dem Dateimonat entsprechen
- Dropoff-Zeitpunkt muss nach Pickup-Zeitpunkt liegen
- Fahrtdauer muss grösser als 0 und höchstens 180 Minuten sein
- Distanz muss grösser als 0 und höchstens 160 km sein
- Durchschnittsgeschwindigkeit muss grösser als 0 und höchstens 150 km/h sein
- Passenger Count muss zwischen 1 und 6 liegen
- Fahrpreis und Trinkgeld dürfen nicht negativ sein
- Gesamtbetrag muss grösser als 0 und höchstens 1000 sein

Für den Dropoff-Zeitpunkt wird nicht geprüft, ob Monat und Jahr exakt zur Datei passen. Eine Fahrt kann kurz vor Mitternacht starten und erst am nächsten Tag oder im nächsten Monat enden.

In [ ]:
Spalten Filterung

In [8]:
required_columns = [
    "pickup_datetime",
    "dropoff_datetime",
    "file_year",
    "file_month",
    "pickup_year",
    "pickup_month",
    "pickup_hour",
    "pickup_weekday",
    "passenger_count",
    "trip_distance_km",
    "trip_duration_minutes",
    "avg_speed_kmh",
    "fare_amount",
    "tip_amount",
    "total_amount",
    "payment_type"
]

numeric_columns = [
    "passenger_count",
    "trip_distance_km",
    "trip_duration_minutes",
    "avg_speed_kmh",
    "fare_amount",
    "tip_amount",
    "total_amount"
]

df_clean = df

# Null-Werte in wichtigen Spalten entfernen
for column in required_columns:
    df_clean = df_clean.filter(F.col(column).isNotNull())

# NaN-Werte in numerischen Spalten entfernen
for column in numeric_columns:
    df_clean = df_clean.filter(~F.isnan(F.col(column)))

# Plausibilitätsfilter anwenden
df_clean = df_clean \
    .filter(F.col("pickup_year") == F.col("file_year")) \
    .filter(F.col("pickup_month") == F.col("file_month")) \
    .filter(F.col("dropoff_datetime") > F.col("pickup_datetime")) \
    .filter(F.col("trip_duration_minutes") > 0) \
    .filter(F.col("trip_duration_minutes") <= 180) \
    .filter(F.col("trip_distance_km") > 0) \
    .filter(F.col("trip_distance_km") <= 160) \
    .filter(F.col("avg_speed_kmh") > 0) \
    .filter(F.col("avg_speed_kmh") <= 150) \
    .filter(F.col("passenger_count").between(1, 6)) \
    .filter(F.col("fare_amount") >= 0) \
    .filter(F.col("tip_amount") >= 0) \
    .filter(F.col("total_amount") > 0) \
    .filter(F.col("total_amount") <= 1000)

In [ ]:
Check und Vergleich nach bereinigung mit den Prepared werten

In [9]:
#Anzahl Records nach bereinigung
records_after = df_clean.count()
records_removed = records_before - records_after
records_removed_pct = records_removed / records_before * 100

print(f"Records before cleaning: {records_before}")
print(f"Records after cleaning: {records_after}")
print(f"Records removed: {records_removed}")
print(f"Records removed in percent: {records_removed_pct:.4f}%")

[Stage 13:=====================================================>(159 + 1) / 160]

Records before cleaning: 633694594
Records after cleaning: 620366041
Records removed: 13328553
Records removed in percent: 2.1033%


In [10]:
# Vergleich before and after
cleaning_summary = spark.createDataFrame(
    [
        ("before_cleaning", records_before),
        ("after_cleaning", records_after),
        ("removed", records_removed)
    ],
    ["step", "record_count"]
)

cleaning_summary.show()

[Stage 17:=============================>                            (2 + 2) / 4]

+---------------+------------+
|           step|record_count|
+---------------+------------+
|before_cleaning|   633694594|
| after_cleaning|   620366041|
|        removed|    13328553|
+---------------+------------+



In [ ]:
Kontrolle der neuen Werte

In [11]:
df_clean.select(
    F.min("pickup_datetime").alias("min_pickup_datetime"),
    F.max("pickup_datetime").alias("max_pickup_datetime"),
    F.min("dropoff_datetime").alias("min_dropoff_datetime"),
    F.max("dropoff_datetime").alias("max_dropoff_datetime"),
    F.min("trip_duration_minutes").alias("min_duration_minutes"),
    F.max("trip_duration_minutes").alias("max_duration_minutes"),
    F.min("trip_distance_km").alias("min_distance_km"),
    F.max("trip_distance_km").alias("max_distance_km"),
    F.min("avg_speed_kmh").alias("min_speed_kmh"),
    F.max("avg_speed_kmh").alias("max_speed_kmh"),
    F.min("passenger_count").alias("min_passenger_count"),
    F.max("passenger_count").alias("max_passenger_count"),
    F.min("total_amount").alias("min_total_amount"),
    F.max("total_amount").alias("max_total_amount")
).show(truncate=False)

[Stage 19:=====================================================>(159 + 1) / 160]

+-------------------+-------------------+--------------------+--------------------+--------------------+--------------------+---------------+---------------+--------------------+-----------------+-------------------+-------------------+----------------+----------------+
|min_pickup_datetime|max_pickup_datetime|min_dropoff_datetime|max_dropoff_datetime|min_duration_minutes|max_duration_minutes|min_distance_km|max_distance_km|min_speed_kmh       |max_speed_kmh    |min_passenger_count|max_passenger_count|min_total_amount|max_total_amount|
+-------------------+-------------------+--------------------+--------------------+--------------------+--------------------+---------------+---------------+--------------------+-----------------+-------------------+-------------------+----------------+----------------+
|2015-01-01 00:00:00|2021-12-31 23:59:51|2015-01-01 00:01:25 |2022-01-01 00:46:40 |0.016666666666666666|180.0               |0.0160934      |159.968396     |0.005376913225058005|149.98722

In [ ]:
Counts bereinigen

In [12]:
df_clean.groupBy("file_year") \
    .agg(F.count("*").alias("record_count")) \
    .orderBy("file_year") \
    .show()

[Stage 22:=====================================================>(159 + 1) / 160]

+---------+------------+
|file_year|record_count|
+---------+------------+
|     2015|   144808589|
|     2016|   129978027|
|     2017|   112251722|
|     2018|   100801269|
|     2019|    81464315|
|     2020|    22889141|
|     2021|    28172978|
+---------+------------+



In [13]:
df_clean.groupBy("file_year", "file_month") \
    .agg(F.count("*").alias("record_count")) \
    .orderBy("file_year", "file_month") \
    .show(100)

[Stage 25:====================================================> (156 + 4) / 160]

+---------+----------+------------+
|file_year|file_month|record_count|
+---------+----------+------------+
|     2015|         1|    12638846|
|     2015|         2|    12340993|
|     2015|         3|    13225708|
|     2015|         4|    12957347|
|     2015|         5|    13048522|
|     2015|         6|    12224181|
|     2015|         7|    11465864|
|     2015|         8|    11033673|
|     2015|         9|    11119358|
|     2015|        10|    12200591|
|     2015|        11|    11201461|
|     2015|        12|    11352045|
|     2016|         1|    10812865|
|     2016|         2|    11283121|
|     2016|         3|    12101305|
|     2016|         4|    11821761|
|     2016|         5|    11730663|
|     2016|         6|    11033993|
|     2016|         7|    10200809|
|     2016|         8|     9853029|
|     2016|         9|    10023762|
|     2016|        10|    10757304|
|     2016|        11|    10006983|
|     2016|        12|    10352432|
|     2017|         1|     9

In [ ]:
Clean Dataset speichern

In [14]:
#Vorherigen Output löschen falls vorhanden
subprocess.run("hdfs dfs -rm -r -f /taxi/clean", shell=True)

CompletedProcess(args='hdfs dfs -rm -r -f /taxi/clean', returncode=0)

In [15]:
#Neu Speichern
df_clean.write \
    .mode("overwrite") \
    .partitionBy("file_year", "file_month") \
    .parquet(CLEAN_PATH)

In [16]:
#Wieder einlesen
df_clean_loaded = spark.read.parquet(CLEAN_PATH)

print("Loaded clean records:", df_clean_loaded.count())
df_clean_loaded.printSchema()

Loaded clean records: 620366041
root
 |-- pickup_datetime: timestamp (nullable = true)
 |-- dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance_miles: double (nullable = true)
 |-- pickup_location_id: long (nullable = true)
 |-- dropoff_location_id: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- pickup_year: integer (nullable = true)
 |-- pickup_month: integer (nullable = true)
 |-- pickup_hour: integer (nullable = true)
 |-- pickup_weekday: integer (nullable = true)
 |-- trip_duration_minutes: double (nullable = true)
 |-- trip_distance_km: double (nullable = true)
 |-- avg_speed_kmh: double (nullable = true)
 |-- tip_percentage: double (nullable = true)
 |-- file_year: integer (nullable = true)
 |-- file_month: integer (nullable = true)



In [ ]:
Checks nach speichern

In [17]:
#HDFS Prüfen
command = "hdfs dfs -du -h -s /taxi/clean"
result = subprocess.check_output(command, shell=True).decode()

print(result)

18.4 G  36.8 G  /taxi/clean



In [18]:
#Sample zeigen
df_clean_loaded.select(
    "pickup_datetime",
    "dropoff_datetime",
    "file_year",
    "file_month",
    "pickup_hour",
    "pickup_weekday",
    "trip_duration_minutes",
    "trip_distance_km",
    "avg_speed_kmh",
    "passenger_count",
    "payment_type",
    "tip_amount",
    "total_amount",
    "tip_percentage"
).show(10, truncate=False)

+-------------------+-------------------+---------+----------+-----------+--------------+---------------------+------------------+------------------+---------------+------------+----------+------------+------------------+
|pickup_datetime    |dropoff_datetime   |file_year|file_month|pickup_hour|pickup_weekday|trip_duration_minutes|trip_distance_km  |avg_speed_kmh     |passenger_count|payment_type|tip_amount|total_amount|tip_percentage    |
+-------------------+-------------------+---------+----------+-----------+--------------+---------------------+------------------+------------------+---------------+------------+----------+------------+------------------+
|2017-11-01 00:01:48|2017-11-01 00:03:47|2017     |11        |0          |4             |1.9833333333333334   |0.6437360000000001|19.474366386554625|1.0            |2           |0.0       |4.8         |0.0               |
|2017-11-01 00:18:22|2017-11-01 00:40:32|2017     |11        |0          |4             |22.166666666666668   |7

In [19]:
spark.stop()